# Marsline · Feature engineering for the Jezero Crater map

**NASA Space Apps 2026 · Interplanetary Survival Guide: Martian Map**

This notebook turns the raw NASA/USGS files in `Marsline_NASA_Resources.zip` into small,
browser-ready layers that the Marsline app loads from `public/data/jezero/`.

| Step | Raw data | Engineered feature | Used in the app for |
|---|---|---|---|
| 1 | HRSC DEM (75 m, in `USGS_Geology`) | **slope, roughness, elevation** | safety, route cost |
| 2 | USGS Jezero/Nili geologic map (`.gdb`) | **terrain-cost + science-value rasters**, GeoJSON overlays | hazards, science targets |
| 3 | CTX mosaic + THEMIS day/night IR | **optical + thermal layers** | context imagery |
| 4 | Steps 1–3 combined | **traversability classes + route cost grid** | route planner (A*) |
| 5 | MOLA global DEM | **cross-check** of HRSC elevations | data-quality validation |
| 6 | MEDA index CSVs | **sol / season / local-time coverage** | conditions page |

Everything is computed in the map's native metric projection (Mars MTM, central meridian 77.5°E),
so slopes are in real degrees, then warped to a regular longitude/latitude grid for the web map.

> **Assumptions are marked `ASSUMPTION`.** Slope limits and geologic-unit costs/science scores are
> engineering choices for this concept, not NASA-certified numbers. Tune them in the config cells.

## 0 · Setup
Needs `geopandas`, `pyogrio`, `pyproj`, `shapely`, `tifffile`, `imagecodecs` in addition to Anaconda's numpy/scipy/pandas/Pillow/matplotlib.
If an import fails, run **one** of these in a terminal, then restart the kernel:
```
conda install -c conda-forge geopandas pyogrio pyproj shapely tifffile imagecodecs
# or
pip install geopandas pyogrio pyproj shapely tifffile imagecodecs
```

In [1]:
import os, json, math, zipfile, shutil, warnings
os.environ["PROJ_IGNORE_CELESTIAL_BODY"] = "YES"   # PROJ otherwise refuses Mars <-> Earth CRS mixing
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from matplotlib import cm
from scipy import ndimage as ndi

try:
    import tifffile, shapely, geopandas as gpd
    from pyproj import CRS, Transformer
    from shapely.geometry import mapping, box
    from shapely.ops import transform as shp_transform
except ImportError as e:
    raise SystemExit(f"Missing package: {e.name}. See the install note above.")

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "package.json").exists())
ZIP_PATH = Path(os.environ.get("MARSLINE_ZIP", Path.home() / "Downloads" / "Marsline_NASA_Resources.zip"))
RAW = ROOT / "data" / "raw"                      # git-ignored (hundreds of MB)
REGIONS_DIR = ROOT / "public" / "data" / "regions"  # one folder per study area, small outputs the app serves
OUT = REGIONS_DIR / "jezero"                       # the detailed Jezero area (steps 1-13)
RAW.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
print("project:", ROOT, "\nzip:", ZIP_PATH, ZIP_PATH.exists())

R_MARS = 3396190.0                                # Mars 2000 sphere radius used by the USGS map (m)
M_PER_DEG = R_MARS * math.pi / 180

# --- ASSUMPTION: mobility limits for a suited astronaut / unpressurised rover ---------------------
SLOPE_BLOCK_DEG = 25.0      # >= this: no-go
SLOPE_CAUTION_DEG = 15.0    # >= this: caution
ROUGH_CAUTION_M = 3.0       # local elevation std-dev (5x5 window) above which terrain is flagged rough

project: /Users/poornapiumini/Documents/Marsline 
zip: /Users/poornapiumini/Downloads/Marsline_NASA_Resources.zip True


## 1 · Unpack only what we need
The zip holds ~600 MB. We extract the MOLA tile and the useful parts of the USGS database
(geodatabase + CTX/THEMIS/HRSC rasters; the 6 m ORTHO and 20 m DTMs use another projection and are skipped).

In [2]:
def extract_inputs():
    marker = RAW / ".extracted"
    if marker.exists():
        return
    assert ZIP_PATH.exists(), f"Put the zip at {ZIP_PATH} or set MARSLINE_ZIP"
    base = "Marsline_NASA_Resources/"
    with zipfile.ZipFile(ZIP_PATH) as z:
        for n in ["MOLA/megt44n000hb.img", "MOLA/megt44n000hb.lbl", "USGS_Geology/sim3464_database.zip"]:
            z.extract(base + n, RAW / "_tmp")
    shutil.move(RAW / "_tmp" / base / "MOLA", RAW / "MOLA")
    inner = RAW / "_tmp" / base / "USGS_Geology" / "sim3464_database.zip"
    keep = ("Geodatabase/", "CoordinateSystems/", "Rasters/HRSC/nili_h0988", "Rasters/Caltech_CTX_mosaic_feb2018/",
            "Rasters/USGS_THEMIS_IR_controlled_mosaic_oct2018/")
    with zipfile.ZipFile(inner) as z:
        for m in z.namelist():
            rel = m.split("/", 1)[-1]
            if m.startswith("__MACOSX") or m.endswith((".lock", ".ovr", ".DS_Store")) or m.endswith("/"):
                continue
            if rel.startswith(keep):
                z.extract(m, RAW / "usgs")
    shutil.rmtree(RAW / "_tmp")
    marker.write_text("ok")

extract_inputs()
USGS = next((RAW / "usgs").iterdir())
GDB = USGS / "Geodatabase" / "Nili.gdb"
print("geodatabase:", GDB.exists(), "| rasters:", sorted(p.name for p in (USGS / "Rasters").rglob("*.tif")))

geodatabase: True | rasters: ['Nili_CTX_Mosaic_Beta01_Feb2018_MTM.tif', 'nili_THEMIS_DayIR_Controlled_oct2018.tif', 'nili_THEMIS_NightIR_Controlled_oct2018.tif', 'nili_h0988_0000_da4_clip.tif', 'nili_h0988_0000_hillshade_clip.tif']


## 2 · Coordinate systems
The USGS map and its rasters use **Mars MTM (Transverse Mercator, central meridian 77.5°E)**. We analyse in MTM metres
and only convert to longitude/latitude for display.

In [3]:
MTM = CRS.from_wkt(next((USGS / "CoordinateSystems").glob("MTM_clon77_5E.prj")).read_text())
LL = CRS.from_proj4(f"+proj=longlat +R={R_MARS} +no_defs")
to_ll = Transformer.from_crs(MTM, LL, always_xy=True)
to_mtm = Transformer.from_crs(LL, MTM, always_xy=True)

def read_geotiff(path):
    """Return (array, x0, y0, res): top-left corner and square pixel size in MTM metres."""
    with tifffile.TiffFile(path) as t:
        pg = t.pages[0]
        sx = pg.tags["ModelPixelScaleTag"].value[0]; tp = pg.tags["ModelTiepointTag"].value
        return pg.asarray(), tp[3], tp[4], sx

r = USGS / "Rasters"
dem_raw, X0, Y0, RES = read_geotiff(r / "HRSC" / "nili_h0988_0000_da4_clip.tif")
H, W = dem_raw.shape
print(f"HRSC DEM grid {W}x{H} @ {RES:.0f} m | extent {W*RES/1000:.1f} x {H*RES/1000:.1f} km | dtype {dem_raw.dtype} | range {dem_raw.min()}..{dem_raw.max()}")

HRSC DEM grid 753x791 @ 75 m | extent 56.5 x 59.3 km | dtype int16 | range -32768..-1413


## 3 · Terrain features from the HRSC DEM (75 m)
* **Slope** (Horn's method, same as GDAL/ArcGIS) – how steep the ground is for a walking astronaut.
* **Roughness** – local standard deviation of elevation after removing the broad trend (a proxy for rocky/uneven ground).
* **Elevation** – kept for elevation-gain estimates along routes.

In [4]:
dem = dem_raw.astype(np.float32)
bad = dem < -9000                                        # no-data safeguard
if bad.any():
    idx = ndi.distance_transform_edt(bad, return_distances=False, return_indices=True)
    dem = dem[tuple(idx)]

def horn_slope(z, res):
    dzdx = ndi.sobel(z, axis=1) / (8 * res)
    dzdy = ndi.sobel(z, axis=0) / (8 * res)
    return np.degrees(np.arctan(np.hypot(dzdx, dzdy)))

def roughness(z, win=5):
    resid = z - ndi.uniform_filter(z, win * 2 - 1)
    m = ndi.uniform_filter(resid, win); m2 = ndi.uniform_filter(resid ** 2, win)
    return np.sqrt(np.maximum(m2 - m ** 2, 0))

slope = horn_slope(dem, RES)
rough = roughness(dem)
print(f"slope: median {np.median(slope):.1f}°, p95 {np.percentile(slope,95):.1f}°, max {slope.max():.1f}° | "
      f"roughness median {np.median(rough):.2f} m")

slope: median 2.9°, p95 12.7°, max 45.1° | roughness median 1.44 m


## 4 · Geology → terrain cost and science value
The USGS map (SIM 3464) gives geologic units, linear features (ridges, rim crests, delta fronts), surface features and craters.

`ASSUMPTION`: the table below converts each unit to a **terrain-cost multiplier** (loose sand and rugged ground are slower
and riskier on foot) and a **science value 0–1** (how much a mission scientist would want to visit). Edit freely.

In [5]:
UNIT_RULES = {
    #  name                      cost  science  why
    "Jezero fan 1 unit":        (1.1, 1.00, "Delta deposit: water-lain sediments, prime astrobiology target"),
    "Jezero fan 2 unit":        (1.1, 0.95, "Younger delta deposit: stratigraphy of the Jezero lake system"),
    "Nili Planum fan unit":     (1.1, 0.70, "Fan deposit outside the crater"),
    "Upper etched unit":        (1.3, 0.80, "Layered, eroded bedrock: mineral-rich layers"),
    "Lower etched unit":        (1.3, 0.80, "Layered, eroded bedrock: older record"),
    "Jezero floor unit":        (1.0, 0.65, "Crater floor: flat, easy access, igneous/sedimentary mix"),
    "Crater interior unit":     (1.2, 0.55, "Crater interior materials"),
    "Crater rim unit":          (1.5, 0.70, "Uplifted ancient crust (megabreccia): steep, blocky"),
    "Crater ejecta unit":       (1.4, 0.50, "Impact ejecta: boulder-rich"),
    "Nili Planum 1 unit":       (1.2, 0.50, "Ancient plateau bedrock"),
    "Nili Planum 2 unit":       (1.2, 0.50, "Ancient plateau bedrock"),
    "Smooth unit, undivided":   (1.0, 0.30, "Smooth cover: easy walking"),
    "Rugged bright unit":       (1.6, 0.40, "Rough bright bedrock: slow, tripping hazard"),
    "Eolian bedform unit":      (1.8, 0.10, "Dunes/ripples: loose sand, wheel/boot entrapment"),
}
DEFAULT_COST, DEFAULT_SCIENCE = 1.2, 0.2
UNIT_NAMES = list(UNIT_RULES)
FEATURE_COST = {"ridged": 1.8, "dark-colored mantling material": 1.3}          # SurfaceFeatures polygons
BARRIER_LINES = ("ridge crest", "crest of crater rim", "delta lobe front")      # LinearFeatures treated as scarps/obstacles

units = gpd.read_file(GDB, layer="GeoUnits_v1")
units["name"] = units["name"].str.replace("riim", "rim").str.strip()
surf = gpd.read_file(GDB, layer="SurfaceFeatures")
lines = gpd.read_file(GDB, layer="LinearFeatures")
craters = gpd.read_file(GDB, layer="LocationFeatures")
ellipse = gpd.read_file(GDB, layer="LandingEllipse")
print(len(units), "units;", units["name"].nunique(), "unit types;", len(lines), "linear features;", len(craters), "craters")

1304 units; 14 unit types; 382 linear features; 341 craters


In [6]:
def pxy(c):  # MTM -> fractional pixel (col,row), pixel corners
    return ((c[0] - X0) / RES, (Y0 - c[1]) / RES)

def burn_polygons(geoms, values):
    """Rasterise polygons (largest first so small units sit on top) onto the DEM grid. NaN where uncovered."""
    out = np.full((H, W), np.nan, np.float32)
    for i in np.argsort([-g.area for g in geoms]):
        g = geoms[i]
        if g is None or g.is_empty: continue
        for p in (g.geoms if g.geom_type == "MultiPolygon" else [g]):
            minx, miny, maxx, maxy = p.bounds
            c0, c1 = max(int((minx - X0) // RES) - 1, 0), min(int((maxx - X0) // RES) + 2, W)
            r0, r1 = max(int((Y0 - maxy) // RES) - 1, 0), min(int((Y0 - miny) // RES) + 2, H)
            if c1 <= c0 or r1 <= r0: continue
            im = Image.new("L", (c1 - c0, r1 - r0), 0); d = ImageDraw.Draw(im)
            d.polygon([(pxy(c)[0] - c0, pxy(c)[1] - r0) for c in p.exterior.coords], fill=1)
            for h in p.interiors:
                d.polygon([(pxy(c)[0] - c0, pxy(c)[1] - r0) for c in h.coords], fill=0)
            out[r0:r1, c0:c1][np.array(im, bool)] = values[i]
    return out

def burn_lines(geoms, width=2):
    im = Image.new("L", (W, H), 0); d = ImageDraw.Draw(im)
    for g in geoms:
        for ln in (g.geoms if g.geom_type == "MultiLineString" else [g]):
            d.line([pxy(c) for c in ln.coords], fill=1, width=width)
    return np.array(im, bool)

unit_idx = burn_polygons(list(units.geometry), [UNIT_NAMES.index(n) if n in UNIT_NAMES else -1 for n in units["name"]])
unit_idx[unit_idx < 0] = np.nan
cost_tbl = np.array([UNIT_RULES[n][0] for n in UNIT_NAMES]); sci_tbl = np.array([UNIT_RULES[n][1] for n in UNIT_NAMES])
geo_cost = np.where(np.isnan(unit_idx), DEFAULT_COST, cost_tbl[np.nan_to_num(unit_idx, nan=0).astype(int)])
science = np.where(np.isnan(unit_idx), DEFAULT_SCIENCE, sci_tbl[np.nan_to_num(unit_idx, nan=0).astype(int)])

for typ, mult in FEATURE_COST.items():
    sel = surf[surf["TYPE"] == typ]
    if len(sel):
        m = ~np.isnan(burn_polygons(list(sel.geometry), [1.0] * len(sel)))
        geo_cost = np.where(m, np.maximum(geo_cost, mult), geo_cost)

barrier = burn_lines(lines[lines["TYPE"].str.startswith(BARRIER_LINES)].geometry, width=2)
print(f"unit coverage {np.mean(~np.isnan(unit_idx))*100:.0f}% of the DEM area | barrier pixels {barrier.sum()}")

unit coverage 100% of the DEM area | barrier pixels 10160


## 5 · Traversability classes and route cost
* **0 · No-go**: slope ≥ 25° or a ridge/rim/delta-front scarp line.
* **1 · Caution**: slope ≥ 15°, rough ground, or high-cost geology (≥ 1.5×).
* **2 · Nominal**.

`route_cost = geology cost × (1 + (slope/15°)²) × (1 + roughness/3 m)`, and no-go cells are impassable.
The app's A* planner reads this same grid.

In [7]:
cls = np.full((H, W), 2, np.uint8)
cls[(slope >= SLOPE_CAUTION_DEG) | (rough >= ROUGH_CAUTION_M) | (geo_cost >= 1.5)] = 1
cls[(slope >= SLOPE_BLOCK_DEG) | barrier] = 0
route_cost = geo_cost * (1 + (slope / SLOPE_CAUTION_DEG) ** 2) * (1 + rough / ROUGH_CAUTION_M)
share = {k: float((cls == v).mean()) for k, v in {"nogo": 0, "caution": 1, "nominal": 2}.items()}
print({k: f"{v*100:.1f}%" for k, v in share.items()})

{'nogo': '2.3%', 'caution': '30.0%', 'nominal': '67.7%'}


## 6 · Warp everything to a longitude/latitude grid for the web map
Because MTM is slightly rotated against the lon/lat graticule here (meridian convergence), we resample properly instead of just
stretching the image. Output pixel = ~75 m for analysis layers.

In [8]:
xs = np.r_[np.linspace(X0, X0 + W * RES, 60), np.full(60, X0 + W * RES), np.linspace(X0 + W * RES, X0, 60), np.full(60, X0)]
ys = np.r_[np.full(60, Y0), np.linspace(Y0, Y0 - H * RES, 60), np.full(60, Y0 - H * RES), np.linspace(Y0 - H * RES, Y0, 60)]
elon, elat = to_ll.transform(xs, ys)
WEST, EAST, SOUTH, NORTH = float(elon.min()), float(elon.max()), float(elat.min()), float(elat.max())
AOI = box(WEST, SOUTH, EAST, NORTH)

def make_grid(step_m):
    dlat = step_m / M_PER_DEG; dlon = dlat / math.cos(math.radians((SOUTH + NORTH) / 2))
    wd, ht = int(math.ceil((EAST - WEST) / dlon)), int(math.ceil((NORTH - SOUTH) / dlat))
    lon = WEST + (np.arange(wd) + 0.5) * dlon; lat = NORTH - (np.arange(ht) + 0.5) * dlat
    LON, LAT = np.meshgrid(lon, lat)
    X, Y = to_mtm.transform(LON, LAT)
    return dict(w=wd, h=ht, dlon=dlon, dlat=dlat, X=X, Y=Y, lon=lon, lat=lat)

G = make_grid(75)
print(f"bounds  W {WEST:.4f}  E {EAST:.4f}  S {SOUTH:.4f}  N {NORTH:.4f} | grid {G['w']}x{G['h']}")

def grid_bounds(g):
    """Exact outer edges [south, west, north, east] of a grid, so web overlays align pixel-for-pixel."""
    return [NORTH - g["h"] * g["dlat"], WEST, NORTH, WEST + g["w"] * g["dlon"]]

def warp(src, res, grid, x0=None, y0=None, order=1):
    x0 = X0 if x0 is None else x0; y0 = Y0 if y0 is None else y0
    col = (grid["X"] - x0) / res - 0.5; row = (y0 - grid["Y"]) / res - 0.5
    return ndi.map_coordinates(src.astype(np.float32), [row, col], order=order, mode="constant", cval=np.nan)

w_dem, w_slope, w_rough = (warp(a, RES, G) for a in (dem, slope, rough))
w_cls = warp(cls.astype(np.float32), RES, G, order=0)
w_geo, w_sci = warp(geo_cost, RES, G), warp(science, RES, G)
w_bar = warp(barrier.astype(np.float32), RES, G, order=0)
w_cost = warp(route_cost, RES, G)
w_bad = warp(bad.astype(np.float32), RES, G, order=0)          # DEM no-data (about 0.4 %) -> invalid
valid = ~np.isnan(w_dem) & (w_bad < 0.5)

bounds  W 76.7960  E 77.8018  S 17.6992  N 18.7013 | grid 756x792


## 7 · Optical and thermal layers (CTX, THEMIS day/night, HRSC hillshade)

In [9]:
def gray_png(a, valid, path, lo=None, hi=None, jpeg=False):
    lo = np.nanpercentile(a, 1) if lo is None else lo; hi = np.nanpercentile(a, 99) if hi is None else hi
    g = np.clip((np.nan_to_num(a) - lo) / (hi - lo + 1e-9), 0, 1)
    rgba = np.dstack([g, g, g, valid]).astype(np.float32); rgba[..., :3] *= 255; rgba[..., 3] *= 255
    im = Image.fromarray(rgba.astype(np.uint8), "RGBA")
    (im.convert("RGB").save(path, quality=80) if jpeg else im.save(path, optimize=True))

def tif(name): return read_geotiff(next(r.rglob(name)))

a, x0, y0, res = tif("nili_h0988_0000_hillshade_clip.tif")
w_hill = warp(a, res, G, x0, y0); gray_png(w_hill, ~np.isnan(w_hill), OUT / "hillshade.png", 0, 255)

a, x0, y0, res = tif("nili_THEMIS_DayIR_Controlled_oct2018.tif")
w_day = warp(a, res, G, x0, y0, order=1); gray_png(w_day, ~np.isnan(w_day), OUT / "themis_day.png")
a, x0, y0, res = tif("nili_THEMIS_NightIR_Controlled_oct2018.tif")
w_night = warp(a, res, G, x0, y0, order=1); gray_png(w_night, ~np.isnan(w_night), OUT / "themis_night.png")

a, x0, y0, res = tif("Nili_CTX_Mosaic_Beta01_Feb2018_MTM.tif")          # 5 m -> average to 20 m
k = 4; a = a[: a.shape[0] // k * k, : a.shape[1] // k * k]
a = a.reshape(a.shape[0] // k, k, a.shape[1] // k, k).mean(axis=(1, 3), dtype=np.float32)
GC = make_grid(20)
w_ctx = warp(a, res * k, GC, x0, y0); gray_png(w_ctx, ~np.isnan(w_ctx), OUT / "ctx.jpg", jpeg=True)
print("THEMIS day/night, hillshade, CTX written; CTX grid", GC["w"], "x", GC["h"])

THEMIS day/night, hillshade, CTX written; CTX grid 2832 x 2970


## 8 · Colour layers + data textures for the app
* `elevation.png`, `slope.png`, `traversability.png`, `science.png` – coloured display layers.
* `grid_data.png` – a **data texture** the route planner decodes in the browser:
  `R = slope × 4`, `G = route cost × 40`, `B = class (0 no-go · 1 caution · 2 nominal · 255 outside)`.
* `grid_elev.png` – elevation: `(elevation + 4096) × 8` split across R (high byte) and G (low byte).

In [10]:
def colorize(a, cmap, lo, hi, alpha=1.0):
    t = np.clip((np.nan_to_num(a, nan=lo) - lo) / (hi - lo), 0, 1)
    rgba = (plt.get_cmap(cmap)(t) * 255).astype(np.uint8)
    rgba[..., 3] = np.where(np.isnan(a), 0, int(alpha * 255)); return Image.fromarray(rgba, "RGBA")

zlo, zhi = np.nanpercentile(w_dem, [1, 99])
colorize(w_dem, "terrain", zlo, zhi).save(OUT / "elevation.png", optimize=True)
colorize(w_slope, "YlOrRd", 0, 35, 0.85).save(OUT / "slope.png", optimize=True)
colorize(w_sci, "viridis", 0, 1, 0.8).save(OUT / "science.png", optimize=True)
# Radiation context: relative atmospheric depth. Lower ground has more atmosphere overhead, which shields a little more from cosmic rays.
H_SCALE = 10_800.0                                    # ASSUMPTION: Mars atmospheric scale height (about 10.8 km)
atm = np.exp(-w_dem / H_SCALE); atm_lo, atm_hi = np.nanpercentile(atm, [1, 99])
colorize(atm, "PuBu", atm_lo, atm_hi, 0.8).save(OUT / "atmosphere.png", optimize=True)
ATM_NOTE = f"Pressure here is {np.nanmin(atm):.2f} to {np.nanmax(atm):.2f} times the reference level (areoid). Darker = more atmosphere overhead. Context only, not a dose."

palette = np.array([[220, 38, 38, 190], [245, 158, 11, 150], [34, 197, 94, 70]], np.uint8)       # no-go / caution / nominal
ci = np.nan_to_num(w_cls, nan=2).astype(int); trav = palette[ci]; trav[~valid] = 0
Image.fromarray(trav, "RGBA").save(OUT / "traversability.png", optimize=True)

data = np.zeros((G["h"], G["w"], 4), np.uint8)
data[..., 0] = np.clip(np.nan_to_num(w_slope) * 4, 0, 255)
data[..., 1] = np.clip(np.nan_to_num(w_cost, nan=9) * 40, 0, 255)
data[..., 2] = np.where(valid, np.nan_to_num(w_cls, nan=2), 255)
data[..., 3] = 255
Image.fromarray(data, "RGBA").save(OUT / "grid_data.png", optimize=True)
enc = np.clip((np.nan_to_num(w_dem) + 4096) * 8, 0, 65535).astype(np.uint16)
elev = np.dstack([enc >> 8, enc & 255, np.zeros_like(enc), np.full_like(enc, 255)]).astype(np.uint8)
Image.fromarray(elev, "RGBA").save(OUT / "grid_elev.png", optimize=True)

fig, ax = plt.subplots(2, 3, figsize=(15, 9))
ext = [WEST, EAST, SOUTH, NORTH]
for a_, im_, t_ in zip(ax.ravel(),
    [w_dem, w_slope, w_rough, np.nan_to_num(w_cls, nan=np.nan), w_sci, w_night],
    ["Elevation (m)", "Slope (°)", "Roughness (m)", "Traversability 0/1/2", "Science value", "THEMIS night IR"]):
    s = a_.imshow(im_, extent=ext, cmap="viridis" if "Sci" in t_ else None); a_.set_title(t_); plt.colorbar(s, ax=a_, shrink=.7)
plt.tight_layout(); plt.show()

## 9 · Vector overlays (GeoJSON) and science targets

In [11]:
def clean(g):
    g = shp_transform(to_ll.transform, g)
    g = g.intersection(AOI)
    return None if g.is_empty else shapely.set_precision(g.simplify(0.0003, preserve_topology=True), 1e-5)

def write_geojson(name, gdf, props, minarea=0):
    feats = []
    for _, row in gdf.iterrows():
        g = clean(row.geometry)
        if g is None or (minarea and g.geom_type.endswith("Polygon") and g.area < minarea): continue
        feats.append({"type": "Feature", "properties": {p: (row[p] if not isinstance(row[p], (np.generic,)) else row[p].item()) for p in props},
                      "geometry": mapping(g)})
    (OUT / name).write_text(json.dumps({"type": "FeatureCollection", "features": feats}, separators=(",", ":")))
    print(f"{name:22s} {len(feats):5d} features {(OUT / name).stat().st_size / 1024:7.0f} KB")

tab = plt.get_cmap("tab20")
units["color"] = [("#%02x%02x%02x" % tuple(int(c * 255) for c in tab(UNIT_NAMES.index(n) % 20)[:3])) if n in UNIT_NAMES else "#888888" for n in units["name"]]
units["cost"] = [UNIT_RULES.get(n, (DEFAULT_COST,))[0] for n in units["name"]]
units["science"] = [UNIT_RULES.get(n, (0, DEFAULT_SCIENCE))[1] for n in units["name"]]
units["note"] = [UNIT_RULES.get(n, ("", "", "Unclassified"))[2] for n in units["name"]]
write_geojson("geology_units.geojson", units, ["name", "color", "cost", "science", "note"], minarea=2e-7)
write_geojson("linear_features.geojson", lines, ["TYPE"])
write_geojson("surface_features.geojson", surf, ["TYPE"], minarea=2e-7)
write_geojson("craters.geojson", craters, ["TYPE"])
ellipse["label"] = ellipse["name"].astype(str)
write_geojson("landing_ellipse.geojson", ellipse, ["label"])

geology_units.geojson   1304 features    1603 KB
linear_features.geojson   382 features      84 KB


surface_features.geojson   285 features     372 KB
craters.geojson          341 features      39 KB
landing_ellipse.geojson     1 features       1 KB


In [12]:
# Science targets: for every high-value unit pick the point deepest inside it that is still walkable.
targets = []
interior = np.zeros((H, W), bool); interior[12:-12, 12:-12] = True       # keep targets ~1 km inside the data edge
for k, nm in enumerate(UNIT_NAMES):
    cost, sci, why = UNIT_RULES[nm]
    if sci < 0.6: continue
    m = (unit_idx == k) & (cls > 0) & interior
    if m.sum() < 40: continue
    dist = ndi.distance_transform_edt(m)
    rr, cc = np.unravel_index(np.argmax(dist), dist.shape)
    lon, lat = to_ll.transform(X0 + (cc + .5) * RES, Y0 - (rr + .5) * RES)
    targets.append(dict(id=f"T{len(targets)+1}", name=nm.replace(" unit", ""), unit=nm, lat=round(lat, 5), lon=round(lon, 5),
                        science=sci, slopeDeg=round(float(slope[rr, cc]), 1), elevationM=round(float(dem[rr, cc])),
                        terrainClass=int(cls[rr, cc]), reason=why))
cx, cy = ellipse.geometry.iloc[0].centroid.coords[0]
lon0, lat0 = to_ll.transform(cx, cy)
start = dict(id="BASE", name="Landing ellipse centre (start)", lat=round(lat0, 5), lon=round(lon0, 5))
targets.sort(key=lambda t: -t["science"])
pd.DataFrame(targets)[["id", "name", "lat", "lon", "science", "slopeDeg", "elevationM", "terrainClass"]]

,id,name,lat,lon,science,slopeDeg,elevationM,terrainClass
0,T1,Jezero fan 1,18.62347,77.56869,1.00,0.2,-2475,2
1,T2,Jezero fan 2,18.48302,77.41788,0.95,2.6,-2456,2
2,T4,Upper etched,18.62972,77.31633,0.80,1.0,-2090,1
3,T5,Lower etched,18.24501,77.74174,0.80,1.6,-2723,2
4,T3,Nili Planum fan,18.60746,76.93586,0.70,4.7,-1890,1
5,T7,Crater rim,18.41964,77.27788,0.70,11.4,-1752,1
6,T6,Jezero floor,18.46909,77.59398,0.65,0.7,-2606,2


## 10 · Cross-check with MOLA (global DEM, 463 m)
HRSC is local and sharp; MOLA is the global reference. If both agree on absolute elevation, the layers are trustworthy.
MOLA tile `MEGT44N000HB`: lat 0–44°N, lon 0–90°E, 128 px/°, big-endian int16, metres above the areoid.

In [13]:
mola = np.fromfile(RAW / "MOLA" / "megt44n000hb.img", dtype=">i2").reshape(5632, 11520)
col = G["lon"][None, :].repeat(G["h"], 0) * 128 - 0.5; row = (44 - G["lat"][:, None].repeat(G["w"], 1)) * 128 - 0.5
w_mola = ndi.map_coordinates(mola.astype(np.float32), [row, col], order=1)
diff = (w_dem - w_mola)[valid]
mola_check = dict(meanDiffM=round(float(np.nanmean(diff)), 1), rmsM=round(float(np.sqrt(np.nanmean(diff ** 2))), 1),
                  corr=round(float(np.corrcoef(w_dem[valid], w_mola[valid])[0, 1]), 3))
print("HRSC − MOLA:", mola_check)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].imshow(w_mola, extent=ext, cmap="terrain"); ax[0].set_title("MOLA (463 m)")
ax[1].imshow(w_dem, extent=ext, cmap="terrain"); ax[1].set_title("HRSC (75 m)"); plt.show()

HRSC − MOLA: {'meanDiffM': 3.3, 'rmsM': 30.8, 'corr': 0.995}


## 11 · MEDA index features (which sols / seasons / hours have data)
The two MEDA CSVs in the zip are **PDS product indexes**, not measurements. We engineer *coverage* features from them
(solar longitude Ls → season, local mean solar time → hour) so the app knows which observations exist
before fetching the real data files by `URN`.

In [14]:
def season(ls): return "Northern spring" if ls < 90 else "Northern summer" if ls < 180 else "Northern autumn" if ls < 270 else "Northern winter"
def hour(s):                                           # "00001M16:02:05.959" -> 16.03
    try: h, m, sec = s.split("M")[1].split(":"); return round(int(h) + int(m) / 60 + float(sec) / 3600, 2)
    except Exception: return None

meda = {}
with zipfile.ZipFile(ZIP_PATH) as z:
    for key, member in {"ancillary": "MEDA/MARS2020_MEDA_ANC_Derived_Del10.csv", "wind": "MEDA/MARS2020+MEDA_WIND_Derived_del-1-8.csv"}.items():
        df = pd.read_csv(z.open("Marsline_NASA_Resources/" + member), skipinitialspace=True)
        df.columns = [c.strip() for c in df.columns]
        ls = pd.to_numeric(df["start_solar_longitude"], errors="coerce")
        meda[key] = pd.DataFrame({
            "sol": pd.to_numeric(df["start_sol_number"], errors="coerce"), "ls": ls.round(1),
            "season": ls.map(lambda v: season(v) if pd.notna(v) else None),
            "startHour": df["start_local_mean_solar_time"].map(hour), "stopHour": df["stop_local_mean_solar_time"].map(hour),
            "sizeMB": df["File Size (MB)"], "urn": df["URN"]}).dropna(subset=["sol"])
        meda[key]["sol"] = meda[key]["sol"].astype(int)
    for k_, d_ in meda.items(): print(k_, len(d_), "sols", d_["sol"].min(), "→", d_["sol"].max())
cover = {k_: {"sols": len(d_), "bySeason": d_["season"].value_counts().to_dict(),
              "rows": d_.where(d_.notna(), None).to_dict("records")} for k_, d_ in meda.items()}
(OUT / "meda_coverage.json").write_text(json.dumps(cover, separators=(",", ":")))
print("meda_coverage.json", (OUT / "meda_coverage.json").stat().st_size // 1024, "KB")

ancillary 1063 sols 1 → 1139
wind 265 sols 36 → 315
meda_coverage.json 251 KB


## 11b · Real MEDA measurements: a typical day in each season
The index files only say *which* data exist. Here we use the real measurements. `scripts/download_meda.py` fetches one sol per season
(Ls ≈ 30°, 120°, 210°, 300°, Mars year 36) from NASA PDS (about 47 MB, saved in `data/raw/meda/`, git-ignored). We reduce each sol to
24 hourly values of local mean solar time and save a ~50 KB JSON that **is** committed, so the app runs without the raw files.

| Quantity | Instrument file | Column |
|---|---|---|
| Pressure (Pa) | `DER_PS` | PRESSURE |
| Local temperature (K) | `DER_RHS` | HUMIDITY_LOCAL_TEMP |
| Relative humidity (%) | `DER_RHS` | LOCAL_RELATIVE_HUMIDITY |
| Ground brightness temperature (K) | `DER_TIRS` | from UPWARD_LW_IRRADIANCE, T = (E / σ)^¼ (emissivity 1: an approximation) |
| Wind speed (m/s) and direction (°) | `DER_WS` | HORIZONTAL_WIND_SPEED, WIND_DIRECTION (only sols 51 and 247 have wind) |

These are measured **at the rover's location**, not along any proposed route.

In [15]:
import subprocess, sys
MEDA_RAW = RAW / "meda"
SIGMA = 5.670374419e-8
SEASON_SOLS = {51: 30, 247: 120, 410: 210, 552: 300}
need = [n for s in SEASON_SOLS for n in (MEDA_RAW / f"sol_{s:04d}").glob("*.CSV")]
if len(need) < 14:
    subprocess.run([sys.executable, str(ROOT / "scripts" / "download_meda.py")], check=True)     # safe to re-run; skips finished files

def lmst_hours(series):
    p = series.str.extract(r"M(\d+):(\d+):([\d.]+)").astype(float)
    return p[0] + p[1] / 60 + p[2] / 3600

def hourly(sol, kind, column, valid, transform=None, agg="median"):
    f = next((MEDA_RAW / f"sol_{sol:04d}").glob(f"*DER_{kind}_*.CSV"), None)
    if f is None: return None
    d = pd.read_csv(f, usecols=["LMST", column] + (["WIND_DIRECTION"] if kind == "WS" and column == "HORIZONTAL_WIND_SPEED" else []))
    d["h"] = lmst_hours(d["LMST"]); v = pd.to_numeric(d[column], errors="coerce")
    d = d.assign(v=v)[valid(v)].dropna(subset=["h", "v"])
    if transform is not None: d["v"] = transform(d["v"])
    d["bin"] = np.floor(d["h"]).astype(int).clip(0, 23)
    g = d.groupby("bin")["v"]
    return {int(b): (round(float(x), 3), int(n)) for b, x, n in zip(g.median().index, g.median(), g.count())}, d

ok = lambda lo, hi: (lambda v: (v > lo) & (v < hi))
sols_out = []
for sol, target_ls in SEASON_SOLS.items():
    row = meda["ancillary"][meda["ancillary"]["sol"] == sol].iloc[0]
    series = {}
    for key, kind, col, valid, tf in [
        ("pressurePa", "PS", "PRESSURE", ok(100, 2000), None),
        ("localTempK", "RHS", "HUMIDITY_LOCAL_TEMP", ok(100, 400), None),
        ("humidityPct", "RHS", "LOCAL_RELATIVE_HUMIDITY", ok(-1, 110), None),
        ("groundTempK", "TIRS", "UPWARD_LW_IRRADIANCE", ok(1, 1000), lambda e: (e / SIGMA) ** 0.25),
        ("windMs", "WS", "HORIZONTAL_WIND_SPEED", ok(-1, 80), None)]:
        res = hourly(sol, kind, col, valid, tf)
        if res is not None: series[key] = res[0]
        if key == "windMs" and res is not None:                                  # circular mean of wind direction per hour
            d = res[1]; dd = pd.read_csv(next((MEDA_RAW / f"sol_{sol:04d}").glob("*DER_WS_*.CSV")), usecols=["LMST", "HORIZONTAL_WIND_SPEED", "WIND_DIRECTION"])
            dd["h"] = lmst_hours(dd["LMST"]); dd["spd"] = pd.to_numeric(dd["HORIZONTAL_WIND_SPEED"], errors="coerce"); dd["dir"] = pd.to_numeric(dd["WIND_DIRECTION"], errors="coerce")
            dd = dd[(dd["spd"] > -1) & (dd["spd"] < 80) & (dd["dir"] >= 0) & (dd["dir"] <= 360)].dropna(); dd["bin"] = np.floor(dd["h"]).astype(int).clip(0, 23)
            r = np.radians(dd["dir"]); dd["sx"], dd["sy"] = np.sin(r), np.cos(r)
            m = dd.groupby("bin")[["sx", "sy"]].mean(); series["windDirDeg"] = {int(b): (round(float((np.degrees(np.arctan2(x.sx, x.sy)) + 360) % 360), 1), 1) for b, x in m.iterrows()}
    hours = []
    for h in range(24):
        item = {"h": h}
        for k, v in series.items():
            if h in v: item[k] = v[h][0]
        hours.append(item)
    def span(k):
        vals = [x[k] for x in hours if k in x]; return [round(min(vals), 2), round(max(vals), 2)] if vals else None
    sols_out.append({"sol": sol, "ls": round(float(row["ls"]), 1), "season": row["season"], "targetLs": target_ls, "hours": hours,
                     "summary": {k: span(k) for k in ["localTempK", "groundTempK", "pressurePa", "humidityPct", "windMs"]}})
meda_ts = {
    "source": "NASA PDS Atmospheres Node · Mars 2020 MEDA derived environmental data (DER_PS, DER_RHS, DER_TIRS, DER_WS)",
    "location": "Perseverance rover, Jezero Crater (measurements are local to the rover)",
    "marsYear": 36, "binning": "hourly medians of local mean solar time (LMST); fill values removed",
    "notes": ["Ground temperature is a brightness temperature from upward long-wave irradiance with emissivity 1 (approximation).",
              "Wind data exist only for sols 51 and 247 in the derived archive; later seasons have no wind file.",
              "One sample sol per season: day-to-day variation, dust storms and instrument flags are not represented."],
    "sols": sols_out}
(OUT / "meda_timeseries.json").write_text(json.dumps(meda_ts, separators=(",", ":")))
print("meda_timeseries.json", round((OUT / "meda_timeseries.json").stat().st_size / 1024), "KB")
for s_ in sols_out: print("sol", s_["sol"], "Ls", s_["ls"], s_["season"], {k: v for k, v in s_["summary"].items() if v})

meda_timeseries.json 10 KB
sol 51 Ls 30.1 Northern spring {'localTempK': [197.14, 254.95], 'groundTempK': [193.98, 272.08], 'pressurePa': [736.1, 755.4], 'windMs': [1.94, 6.97]}
sol 247 Ls 119.8 Northern summer {'localTempK': [199.48, 253.11], 'groundTempK': [199.23, 277.0], 'pressurePa': [645.8, 664.71], 'humidityPct': [0.0, 18.45], 'windMs': [1.05, 8.34]}
sol 410 Ls 209.2 Northern autumn {'localTempK': [196.04, 264.27], 'groundTempK': [187.33, 284.05], 'pressurePa': [692.93, 745.1], 'humidityPct': [0.03, 16.39]}
sol 552 Ls 300.2 Northern winter {'localTempK': [190.71, 250.24], 'groundTempK': [180.04, 266.29], 'pressurePa': [736.43, 776.12], 'humidityPct': [0.04, 71.25]}


## 12 · Write the metadata the app reads

In [16]:
meta = {
    "version": 2, "id": "jezero", "name": "Jezero Crater & delta", "region": "Jezero Crater & delta (Nili Fossae, Mars)", "detail": "detailed",
    "blurb": "Perseverance landing region: delta, crater floor and rim with 75 m terrain, CTX imagery, THEMIS thermal and a USGS geologic map.",
    "vectors": ["geology", "linear", "craters", "ellipse"], "defaultLayers": ["ctx", "traversability", "linear", "ellipse"],
    "elevEncoding": {"offset": 4096, "scale": 8}, "missionMap": "jezero_mission_map.png", "crs": "Mars 2000 sphere, lon/lat east-positive",
    "bounds": {"west": WEST, "east": EAST, "south": SOUTH, "north": NORTH},
    "grid": {"width": G["w"], "height": G["h"], "dlon": G["dlon"], "dlat": G["dlat"], "stepM": 75, "mPerDeg": M_PER_DEG},
    "thresholds": {"slopeBlockDeg": SLOPE_BLOCK_DEG, "slopeCautionDeg": SLOPE_CAUTION_DEG, "roughCautionM": ROUGH_CAUTION_M},
    "encoding": {"grid_data.png": "R=slope*4 (deg), G=route cost*40, B=class 0 no-go/1 caution/2 nominal/255 outside",
                 "grid_elev.png": "(R*256+G)/8 - 4096 = elevation metres"},
    "start": start, "targets": targets, "traversabilityShare": share, "molaCheck": mola_check,
    "elevationRangeM": [round(float(np.nanmin(w_dem))), round(float(np.nanmax(w_dem)))],
    "assumptions": "Slope limits and geologic-unit costs/science scores are concept-level assumptions, not NASA-certified values.",
    "layers": [
        {"id": "ctx", "base": True, "title": "CTX optical mosaic", "file": "ctx.jpg", "dataType": "OPTICAL", "source": "MRO CTX · Caltech mosaic (Feb 2018)", "resolution": "5 m (shown at 20 m)"},
        {"id": "hillshade", "title": "HRSC hillshade", "file": "hillshade.png", "dataType": "TOPOGRAPHIC", "source": "Mars Express HRSC · USGS SIM 3464", "resolution": "75 m"},
        {"id": "elevation", "title": "Elevation", "file": "elevation.png", "dataType": "TOPOGRAPHIC", "source": "Mars Express HRSC DTM (checked vs MGS MOLA)", "resolution": "75 m"},
        {"id": "slope", "title": "Slope", "file": "slope.png", "dataType": "DERIVED", "source": "Derived from HRSC DTM (Horn)", "resolution": "75 m"},
        {"id": "themis_day", "title": "THEMIS day infrared", "file": "themis_day.png", "dataType": "THERMAL", "source": "Mars Odyssey THEMIS controlled mosaic (Oct 2018)", "resolution": "100 m"},
        {"id": "themis_night", "title": "THEMIS night infrared", "file": "themis_night.png", "dataType": "THERMAL", "source": "Mars Odyssey THEMIS controlled mosaic (Oct 2018)", "resolution": "100 m"},
        {"id": "geology", "title": "Geologic units", "file": "geology_units.geojson", "dataType": "GEOLOGIC", "source": "USGS SIM 3464 (Jezero & Nili Fossae)", "resolution": "map scale 1:75,000"},
        {"id": "atmosphere", "title": "Atmospheric depth (radiation shielding context)", "file": "atmosphere.png", "dataType": "DERIVED", "source": "Derived from elevation with an assumed 10.8 km scale height", "resolution": "75 m", "note": ATM_NOTE},
        {"id": "science", "title": "Science value", "file": "science.png", "dataType": "DERIVED", "source": "Derived from USGS units (assumption-based score)", "resolution": "75 m"},
        {"id": "traversability", "title": "Traversability", "file": "traversability.png", "dataType": "DERIVED", "source": "Slope + roughness + geology + scarps", "resolution": "75 m"},
    ],
}
meta["gridBounds"] = grid_bounds(G)
for L_ in meta["layers"]:
    if L_["file"].endswith((".png", ".jpg")):
        L_["bounds"] = grid_bounds(GC if L_["id"] == "ctx" else G)
(OUT / "meta.json").write_text(json.dumps(meta, indent=1))
for p in sorted(OUT.iterdir()): print(f"{p.name:26s}{p.stat().st_size/1024:9.0f} KB")
print("TOTAL", round(sum(p.stat().st_size for p in OUT.iterdir()) / 1e6, 1), "MB")

atmosphere.png                  309 KB
craters.geojson                  39 KB
ctx.jpg                        1794 KB
elevation.png                   330 KB
geology_units.geojson          1603 KB
grid_data.png                   680 KB
grid_elev.png                   622 KB
hillshade.png                   234 KB
jezero_mission_map.png         4547 KB
landing_ellipse.geojson           1 KB
linear_features.geojson          84 KB
meda_coverage.json              252 KB
meda_timeseries.json             10 KB
meta.json                         7 KB
rover                             0 KB
science.png                     306 KB
slope.png                       557 KB
surface_features.geojson        372 KB
themis_day.png                  467 KB
themis_night.png                555 KB
thumbs                            0 KB
traversability.png               53 KB
TOTAL 13.1 MB


## 13 · One-page mission map for the whole study area
A single, printable map of the Jezero area that combines every layer an astronaut needs for a Marswalk:
CTX imagery, terrain safety (slope/roughness/geology → no-go · caution · nominal), geologic boundaries, ridges and rim crests,
science targets, and the least-cost walking routes from the landing site to the two best targets.
Saved as `public/data/jezero/jezero_mission_map.png` (the app offers it as a download).

In [17]:
import matplotlib.patheffects as pe
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from skimage.graph import route_through_array

walkable = (cls > 0) & ~bad
cost_grid = np.where(walkable, route_cost, 1e6).astype(float)
nearest = ndi.distance_transform_edt(~walkable, return_distances=False, return_indices=True)
WALK_M_PER_MIN = 3000 / 60                                     # ASSUMPTION: 3 km/h on easy ground

def plan(a, b):
    def cell(p):
        x, y = to_mtm.transform(p["lon"], p["lat"]); r_, c_ = int(round((Y0 - y) / RES - .5)), int(round((x - X0) / RES - .5))
        return nearest[0][r_, c_], nearest[1][r_, c_]
    path = np.array(route_through_array(cost_grid, cell(a), cell(b), fully_connected=True, geometric=True)[0])
    lon, lat = to_ll.transform(X0 + (path[:, 1] + .5) * RES, Y0 - (path[:, 0] + .5) * RES)
    step = np.hypot(np.diff(path[:, 0]), np.diff(path[:, 1])) * RES
    c = route_cost[path[:, 0], path[:, 1]]
    return dict(lon=lon, lat=lat, km=step.sum() / 1000, minutes=float((step * (c[1:] + c[:-1]) / 2).sum() / WALK_M_PER_MIN),
                maxSlope=float(slope[path[:, 0], path[:, 1]].max()), cautionPct=float((cls[path[:, 0], path[:, 1]] == 1).mean() * 100))

best = sorted(targets, key=lambda t: -t["science"])[:2]
routes = [(t, plan(start, t)) for t in best]
for t, rt in routes: print(f"{start['id']} -> {t['id']} {t['name']}: {rt['km']:.1f} km, {rt['minutes']/60:.1f} h, max slope {rt['maxSlope']:.1f} deg")

BASE -> T1 Jezero fan 1: 13.9 km, 7.2 h, max slope 8.0 deg
BASE -> T2 Jezero fan 2: 1.8 km, 1.1 h, max slope 6.5 deg


In [18]:
def rings(fc, polygons=True):
    out = []
    for f in fc["features"]:
        g = f["geometry"]; t_, c_ = g["type"], g["coordinates"]
        parts = {"Polygon": [c_], "MultiPolygon": c_, "LineString": [[c_]], "MultiLineString": [[l] for l in c_]}.get(t_, [])
        for poly in parts:
            for ring in (poly[:1] if polygons else poly): out.append((np.array(ring), f["properties"]))
    return out
load = lambda n: json.loads((OUT / n).read_text())
LINE_COLORS = {"ridge crest (type 1), certain": "#e0b070", "lineament": "#b8a0ff", "crest of crater rim": "#ff7b6b",
               "delta lobe front": "#5ec8ff", "channel (fluvial)": "#3d8bff", "crest of buried crater": "#9aa5b1"}

fig, ax = plt.subplots(figsize=(13, 15.2))
fig.patch.set_facecolor("#0c1117"); ax.set_facecolor("#0c1117")
sC, wC, nC, eC = grid_bounds(GC); s_, w_, n_, e_ = grid_bounds(G)
ax.imshow(np.ma.masked_invalid(w_ctx), extent=[wC, eC, sC, nC], cmap="gray", vmin=np.nanpercentile(w_ctx, 1), vmax=np.nanpercentile(w_ctx, 99), interpolation="bilinear", zorder=1)
trav_soft = trav.copy(); trav_soft[..., 3] = (trav_soft[..., 3] * 0.55).astype(np.uint8)
ax.imshow(trav_soft, extent=[w_, e_, s_, n_], zorder=2, interpolation="nearest")
ax.add_collection(LineCollection([r for r, _ in rings(load("geology_units.geojson"))], colors="white", linewidths=0.35, alpha=0.35, zorder=3))
for typ, col in LINE_COLORS.items():
    segs = [r for r, p in rings(load("linear_features.geojson"), polygons=False) if p["TYPE"] == typ]
    ax.add_collection(LineCollection(segs, colors=col, linewidths=1.1, alpha=0.95, zorder=4))
for r, _ in rings(load("landing_ellipse.geojson")): ax.plot(r[:, 0], r[:, 1], "--", color="white", lw=1.5, zorder=5)

halo = [pe.withStroke(linewidth=3.2, foreground="#05070a")]
for (t, rt), col, ls in zip(routes, ["#38bdf8", "#f472b6"], ["-", "-"]):
    ax.plot(rt["lon"], rt["lat"], ls, color=col, lw=2.6, zorder=7, path_effects=[pe.withStroke(linewidth=5, foreground="#05070a")])
for t in targets:
    ax.scatter(t["lon"], t["lat"], s=110, c="#f59e0b", edgecolors="#05070a", linewidths=1.6, zorder=8)
    right_edge = t["lon"] > EAST - 0.12                               # keep labels inside the frame
    ax.annotate(f"{t['id']} {t['name']}", (t["lon"], t["lat"]), xytext=(-9 if right_edge else 8, 7), textcoords="offset points", ha="right" if right_edge else "left",
                color="white", fontsize=9, fontweight="bold", path_effects=halo, zorder=9)
ax.scatter(start["lon"], start["lat"], s=150, marker="s", c="white", edgecolors="#05070a", linewidths=1.6, zorder=9)
ax.annotate("BASE · landing ellipse centre", (start["lon"], start["lat"]), xytext=(9, -14), textcoords="offset points", color="white", fontsize=9, fontweight="bold", path_effects=halo, zorder=9)

ax.set_xlim(WEST, EAST); ax.set_ylim(SOUTH, NORTH); ax.set_aspect(1 / math.cos(math.radians((SOUTH + NORTH) / 2)))
ax.tick_params(colors="#c9d1d9", labelsize=9); [s.set_color("#3b4650") for s in ax.spines.values()]
ax.set_xlabel("Longitude (°E)", color="#c9d1d9"); ax.set_ylabel("Latitude (°N)", color="#c9d1d9")
ax.grid(color="white", alpha=0.12, lw=0.5)
bar = 10000 / (M_PER_DEG * math.cos(math.radians((SOUTH + NORTH) / 2))); bx, by = WEST + 0.03, SOUTH + 0.035
ax.plot([bx, bx + bar], [by, by], color="white", lw=3, zorder=10); ax.text(bx + bar / 2, by + 0.01, "10 km", color="white", ha="center", fontsize=10, fontweight="bold", path_effects=halo, zorder=10)
ax.annotate("N", xy=(0.955, 0.965), xytext=(0.955, 0.915), xycoords="axes fraction", color="white", ha="center", fontsize=13, fontweight="bold", arrowprops=dict(arrowstyle="-|>", color="white", lw=2), path_effects=halo, zorder=10)

handles = [Patch(color="#22c55e", alpha=.6, label="Nominal terrain"), Patch(color="#f59e0b", alpha=.7, label="Caution (≥15° slope, rough, high-cost geology)"), Patch(color="#dc2626", alpha=.8, label="No-go (≥25° slope or scarp)"),
           Line2D([], [], color="#ff7b6b", lw=1.6, label="Crater-rim crest"), Line2D([], [], color="#5ec8ff", lw=1.6, label="Delta lobe front"), Line2D([], [], color="#e0b070", lw=1.6, label="Ridge crest"),
           Line2D([], [], marker="o", color="none", markerfacecolor="#f59e0b", markeredgecolor="#05070a", markersize=9, label="Science target"),
           Line2D([], [], color="#38bdf8", lw=2.6, label=f"Route A: BASE → {routes[0][0]['id']}"), Line2D([], [], color="#f472b6", lw=2.6, label=f"Route B: BASE → {routes[1][0]['id']}")]
leg = ax.legend(handles=handles, loc="lower right", facecolor="#101318", edgecolor="#3b4650", labelcolor="white", fontsize=8.5, framealpha=.92); leg.set_zorder(11)

fig.suptitle("JEZERO CRATER · MARSWALK PLANNING MAP", color="white", fontsize=18, fontweight="bold", x=0.07, ha="left", y=0.975)
fig.text(0.07, 0.945, "Perseverance landing region · HRSC elevation · CTX imagery · USGS geologic map (SIM 3464) · MGS MOLA cross-check", color="#929eaa", fontsize=10, ha="left")
lines_ = [f"Route {'AB'[i]}  BASE → {t['id']} {t['name']}:  {rt['km']:.1f} km  ·  about {int(rt['minutes'] // 60)} h {int(rt['minutes'] % 60):02d} min walking  ·  max slope {rt['maxSlope']:.1f}°  ·  {rt['cautionPct']:.0f}% on caution terrain  ·  science value {t['science']:.2f}" for i, (t, rt) in enumerate(routes)]
fig.text(0.07, 0.115, "\n".join(lines_), color="white", fontsize=10, va="top", linespacing=1.7)
fig.text(0.07, 0.065, "Concept-level estimates: walking speed 3 km/h on easy ground; slope limits 15° (caution) / 25° (no-go); geologic-unit costs and science scores are assumptions, not NASA-certified values.\n"
         "Data: NASA/ESA HRSC, NASA MRO CTX (Caltech mosaic), Mars Odyssey THEMIS, USGS SIM 3464, NASA MGS MOLA. Coordinates: Mars 2000 sphere, east-positive. Built by Marsline.", color="#929eaa", fontsize=8, va="top", linespacing=1.6)
fig.subplots_adjust(left=0.07, right=0.97, top=0.925, bottom=0.17)
fig.savefig(OUT / "jezero_mission_map.png", dpi=140, facecolor=fig.get_facecolor())
plt.show()
print("jezero_mission_map.png", round((OUT / "jezero_mission_map.png").stat().st_size / 1e6, 1), "MB")

jezero_mission_map.png 4.7 MB


## 13b · Orbital preview thumbnails for the Jezero targets
A small CTX crop (about 3 km wide) centred on each target, so the crew can see what the ground looks like from above before they go.

In [19]:
from PIL import ImageDraw
tdir = OUT / "thumbs"; tdir.mkdir(exist_ok=True)
meta_j = json.loads((OUT / "meta.json").read_text())
sC, wC, nC, eC = grid_bounds(GC); half = 80                      # 160 px x 20 m = 3.2 km
lo_, hi_ = np.nanpercentile(w_ctx, [1, 99])
for t in meta_j["targets"]:
    c0 = int((t["lon"] - wC) / GC["dlon"]); r0 = int((nC - t["lat"]) / GC["dlat"])
    crop = np.full((2 * half, 2 * half), np.nan, np.float32)
    ra, rb, ca, cb = max(r0 - half, 0), min(r0 + half, w_ctx.shape[0]), max(c0 - half, 0), min(c0 + half, w_ctx.shape[1])
    crop[ra - (r0 - half): rb - (r0 - half), ca - (c0 - half): cb - (c0 - half)] = w_ctx[ra:rb, ca:cb]
    g = (np.clip((np.nan_to_num(crop, nan=lo_) - lo_) / (hi_ - lo_), 0, 1) * 255).astype(np.uint8)
    im = Image.fromarray(g).resize((480, 480), Image.BILINEAR).convert("RGB"); d = ImageDraw.Draw(im)
    d.ellipse([240 - 14, 240 - 14, 240 + 14, 240 + 14], outline=(245, 158, 11), width=3)
    d.text((10, 10), f"{t['id']} {t['name']} · CTX · 3.2 km", fill=(255, 255, 255))
    im.save(tdir / f"{t['id']}.jpg", quality=82); t["thumb"] = f"thumbs/{t['id']}.jpg"
(OUT / "meta.json").write_text(json.dumps(meta_j, indent=1))
print("thumbnails:", [p.name for p in sorted(tdir.iterdir())], round(sum(p.stat().st_size for p in tdir.iterdir()) / 1024), "KB")

thumbnails: ['T1.jpg', 'T2.jpg', 'T3.jpg', 'T4.jpg', 'T5.jpg', 'T6.jpg', 'T7.jpg'] 204 KB


## 14 · More study areas (regional scale, from the global MOLA elevation tile)
The zip contains **one global elevation tile** (MOLA, 128 px/° ≈ 463 m, lon 0–90 °E, lat 0–44 °N). From it we can build the same
safety layers for any other area in that window: hillshade, slope, roughness, no-go/caution/nominal terrain, relief-derived science
targets, and routes. Add your own area by adding one line to `REGIONS` below and re-running.

Differences from Jezero (be honest in the app and the pitch):
* **463 m cells**, so local steep ground is smoothed out. We therefore use lower slope limits (8° caution, 15° no-go) – `ASSUMPTION`.
* **No geologic map, CTX or HRSC** for these areas, so targets are *relief-derived candidates*, not mapped rock units.
* The colour and THEMIS PNG mosaics in the zip carry no map coordinates, so they are not used here.

In [20]:
from matplotlib.colors import LightSource
REG_SLOPE_CAUTION, REG_SLOPE_BLOCK = 8.0, 15.0     # ASSUMPTION for 463 m cells
BASE_RADIUS_M, REACH_M = 20_000, 18_000             # ASSUMPTION: base within 20 km of the area centre; targets within 18 km of the base

# lat/lon = centre of the area, half = half-height in degrees (width is widened so the box is roughly square in km).
# To add an area: append a dict. It must lie inside lon 0-90 E and lat 0-44 N (the MOLA tile you have).
REGIONS = [
    dict(id="nili-fossae", name="Nili Fossae", lat=21.5, lon=74.5, half=1.5,
         blurb="Arcuate trough system just north-west of Jezero. Orbiters detect clay, olivine and carbonate minerals here."),
    dict(id="syrtis-major", name="Syrtis Major · Nili Patera", lat=8.8, lon=67.3, half=1.5,
         blurb="Dark volcanic highland with the Nili Patera caldera, a dune field and silica-rich deposits seen from orbit.",
         landmarks=[dict(name="Nili Patera caldera", lat=8.8, lon=67.3, reason="Volcanic caldera at the centre of this area")]),
    dict(id="isidis-planitia", name="Isidis Planitia", lat=12.9, lon=87.0, half=2.5,
         blurb="Smooth floor of a giant impact basin about 1,500 km across: broad flat terrain, a natural candidate for landing studies."),
]
lightsrc = LightSource(azdeg=315, altdeg=45)

def build_region(spec):
    rid = spec["id"]; out = REGIONS_DIR / rid; out.mkdir(parents=True, exist_ok=True)
    hl = spec.get("half", 1.5); hlon = hl / math.cos(math.radians(spec["lat"]))
    lon0, lon1, lat0, lat1 = spec["lon"] - hlon, spec["lon"] + hlon, spec["lat"] - hl, spec["lat"] + hl
    if not (0 <= lon0 and lon1 <= 90 and 0 <= lat0 and lat1 <= 44):
        raise ValueError(f"{rid}: the MOLA tile only covers lon 0-90E, lat 0-44N")
    r0, r1 = int((44 - lat1) * 128), int(math.ceil((44 - lat0) * 128)); c0, c1 = int(lon0 * 128), int(math.ceil(lon1 * 128))
    z = mola[r0:r1, c0:c1].astype(np.float32); hh, ww = z.shape
    west, east, north, south = c0 / 128, c1 / 128, 44 - r0 / 128, 44 - r1 / 128
    dy = M_PER_DEG / 128; dx = dy * math.cos(math.radians(spec["lat"]))
    sl = np.degrees(np.arctan(np.hypot(ndi.sobel(z, axis=1) / (8 * dx), ndi.sobel(z, axis=0) / (8 * dy))))
    ro = roughness(z, 3); rough_caution = float(max(np.percentile(ro, 92), 10))
    rcls = np.full(z.shape, 2, np.uint8); rcls[(sl >= REG_SLOPE_CAUTION) | (ro >= rough_caution)] = 1; rcls[sl >= REG_SLOPE_BLOCK] = 0
    rcost = (1 + (sl / REG_SLOPE_CAUTION) ** 2) * (1 + ro / rough_caution)

    # ---- display layers --------------------------------------------------------------------------------------------
    hill = lightsrc.hillshade(z, vert_exag=4, dx=dx, dy=dy)
    gray_png(hill, np.ones_like(hill, bool), out / "hillshade.png", 0, 1)
    zlo, zhi = np.percentile(z, [1, 99])
    colorize(z, "terrain", zlo, zhi, 0.9).save(out / "elevation.png", optimize=True)
    colorize(sl, "YlOrRd", 0, REG_SLOPE_BLOCK * 1.4, 0.85).save(out / "slope.png", optimize=True)
    atm_r = np.exp(-z / H_SCALE); alo, ahi = np.percentile(atm_r, [1, 99])
    colorize(atm_r, "PuBu", alo, ahi, 0.8).save(out / "atmosphere.png", optimize=True)
    atm_note = f"Pressure here is {atm_r.min():.2f} to {atm_r.max():.2f} times the reference level (areoid). Darker = more atmosphere overhead. Context only, not a dose."
    trav_r = palette[rcls]; Image.fromarray(trav_r, "RGBA").save(out / "traversability.png", optimize=True)
    data = np.dstack([np.clip(sl * 4, 0, 255), np.clip(rcost * 40, 0, 255), rcls, np.full(z.shape, 255)]).astype(np.uint8)
    Image.fromarray(data, "RGBA").save(out / "grid_data.png", optimize=True)
    enc = np.clip((z + 8192) * 4, 0, 65535).astype(np.uint16)
    Image.fromarray(np.dstack([enc >> 8, enc & 255, np.zeros_like(enc), np.full_like(enc, 255)]).astype(np.uint8), "RGBA").save(out / "grid_elev.png", optimize=True)

    # ---- relief-derived targets ---------------------------------------------------------------------------------------
    inner = np.zeros(z.shape, bool); inner[8:-8, 8:-8] = True
    walk = (rcls > 0) & inner
    ll = lambda r, c: (round(west + (c + .5) / 128, 5), round(north - (r + .5) / 128, 5))
    cell_info = lambda r, c: dict(slopeDeg=round(float(sl[r, c]), 1), elevationM=round(float(z[r, c])), terrainClass=int(rcls[r, c]))
    found = []
    def add(name, r, c, sci, reason, unit="relief-derived"):
        if any(math.hypot(r - r_, c - c_) < 6 for r_, c_ in [f["_rc"] for f in found]): return
        lon, lat = ll(r, c); found.append(dict(_rc=(r, c), name=name, unit=unit, lat=lat, lon=lon, science=sci, reason=reason, **cell_info(r, c)))
    yy, xx = np.mgrid[0:hh, 0:ww]                                      # distances in metres from a cell
    km_from = lambda r, c: np.hypot((yy - r) * dy, (xx - c) * dx)
    near_centre = km_from((north - spec["lat"]) * 128, (spec["lon"] - west) * 128) < BASE_RADIUS_M
    flat = (sl < 1.5) & walk & near_centre
    if not flat.any(): flat = walk & near_centre
    dist = ndi.distance_transform_edt(flat); br, bc = np.unravel_index(np.argmax(dist), dist.shape)
    walk = walk & (km_from(br, bc) < REACH_M)                           # targets must be within a day's reach of the base
    blon, blat = ll(br, bc); start_ = dict(id="BASE", name="Flattest wide plain (base candidate)", lat=blat, lon=blon)
    found.append(dict(_rc=(br, bc), name="BASE", unit="", lat=blat, lon=blon, science=0, reason="", **cell_info(br, bc)))
    for lm in spec.get("landmarks", []):
        c = int(round((lm["lon"] - west) * 128 - .5)); r = int(round((north - lm["lat"]) * 128 - .5))
        if km_from(br, bc)[r, c] > 30_000: continue
        add(lm["name"], r, c, 0.8, lm["reason"], unit="landmark")
    low = np.where((sl < 3) & walk, z, np.inf)
    if np.isfinite(low.min()): r, c = np.unravel_index(np.argmin(low), low.shape); add("Lowest walkable floor", r, c, 0.7, "Deepest flat ground in the area: basin-floor deposits collect here")
    high = np.where(walk, z, -np.inf); r, c = np.unravel_index(np.argmax(high), high.shape); add("Highest walkable ground", r, c, 0.6, "Rim/summit viewpoint with the widest exposure of older terrain")
    steep = np.where((rcls == 1) & (sl < REG_SLOPE_BLOCK) & inner, sl, -1)
    if steep.max() > 0: r, c = np.unravel_index(np.argmax(steep), steep.shape); add("Steepest walkable slope", r, c, 0.5, "Escarpment where layering may be exposed (confirm with imagery)")
    tg = [{k: v for k, v in f.items() if k != "_rc"} for f in found[1:]]
    tg.sort(key=lambda t: -t["science"])
    for i, t in enumerate(tg): t["id"] = f"T{i + 1}"

    # ---- A* routes from the base to the two best targets ---------------------------------------------------------------
    pas = rcls > 0; cg = np.where(pas, rcost, 1e6).astype(float); nn = ndi.distance_transform_edt(~pas, return_distances=False, return_indices=True)
    def plan_r(a, b):
        cell = lambda p: (nn[0][min(max(int(round((north - p["lat"]) * 128 - .5)), 0), hh - 1), min(max(int(round((p["lon"] - west) * 128 - .5)), 0), ww - 1)],
                          nn[1][min(max(int(round((north - p["lat"]) * 128 - .5)), 0), hh - 1), min(max(int(round((p["lon"] - west) * 128 - .5)), 0), ww - 1)])
        path = np.array(route_through_array(cg, cell(a), cell(b), fully_connected=True, geometric=True)[0])
        step = np.hypot(np.diff(path[:, 0]) * dy, np.diff(path[:, 1]) * dx); cc = rcost[path[:, 0], path[:, 1]]
        return dict(lon=west + (path[:, 1] + .5) / 128, lat=north - (path[:, 0] + .5) / 128, km=step.sum() / 1000,
                    minutes=float((step * (cc[1:] + cc[:-1]) / 2).sum() / WALK_M_PER_MIN), maxSlope=float(sl[path[:, 0], path[:, 1]].max()),
                    cautionPct=float((rcls[path[:, 0], path[:, 1]] == 1).mean() * 100))
    rts = [(t, plan_r(start_, t)) for t in tg[:2]]

    meta_r = {
        "version": 2, "id": rid, "name": spec["name"], "region": spec["name"], "detail": "regional", "blurb": spec["blurb"],
        "vectors": [], "defaultLayers": ["hillshade", "elevation", "traversability"], "elevEncoding": {"offset": 8192, "scale": 4}, "missionMap": "mission_map.png",
        "gridBounds": [south, west, north, east], "bounds": {"west": west, "east": east, "south": south, "north": north},
        "grid": {"width": ww, "height": hh, "dlon": 1 / 128, "dlat": 1 / 128, "stepM": round(dy), "mPerDeg": M_PER_DEG},
        "thresholds": {"slopeBlockDeg": REG_SLOPE_BLOCK, "slopeCautionDeg": REG_SLOPE_CAUTION, "roughCautionM": round(rough_caution, 1)},
        "encoding": {"grid_data.png": "R=slope*4 (deg), G=route cost*40, B=class 0 no-go/1 caution/2 nominal", "grid_elev.png": "(R*256+G)/4 - 8192 = elevation metres"},
        "start": start_, "targets": tg, "traversabilityShare": {k: float((rcls == v).mean()) for k, v in {"nogo": 0, "caution": 1, "nominal": 2}.items()},
        "molaCheck": None, "elevationRangeM": [round(float(z.min())), round(float(z.max()))],
        "assumptions": "Regional 463 m MOLA cells: slope limits (8 deg caution, 15 deg no-go), science scores and walking speed are concept-level assumptions. Targets are relief-derived candidates, not mapped geology.",
        "layers": [
            {"id": "hillshade", "base": True, "title": "MOLA hillshade", "file": "hillshade.png", "dataType": "TOPOGRAPHIC", "source": "MGS MOLA 128 ppd (computed hillshade)", "resolution": "463 m", "bounds": [south, west, north, east]},
            {"id": "elevation", "title": "Elevation", "file": "elevation.png", "dataType": "TOPOGRAPHIC", "source": "MGS MOLA gridded topography", "resolution": "463 m", "bounds": [south, west, north, east]},
            {"id": "slope", "title": "Slope", "file": "slope.png", "dataType": "DERIVED", "source": "Derived from MOLA (Horn)", "resolution": "463 m", "bounds": [south, west, north, east]},
            {"id": "atmosphere", "title": "Atmospheric depth (radiation shielding context)", "file": "atmosphere.png", "dataType": "DERIVED", "source": "Derived from MOLA elevation with an assumed 10.8 km scale height", "resolution": "463 m", "note": atm_note, "bounds": [south, west, north, east]},
            {"id": "traversability", "title": "Traversability", "file": "traversability.png", "dataType": "DERIVED", "source": "Slope + roughness (regional limits)", "resolution": "463 m", "bounds": [south, west, north, east]},
        ],
    }
    (out / "meta.json").write_text(json.dumps(meta_r, indent=1))

    # ---- one-page map for this area ------------------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(11, 12.2)); fig.patch.set_facecolor("#0c1117"); ax.set_facecolor("#0c1117")
    ext = [west, east, south, north]
    ax.imshow(hill, extent=ext, cmap="gray", vmin=0, vmax=1, zorder=1)
    ax.imshow(np.asarray(colorize(z, "terrain", zlo, zhi, 0.5)), extent=ext, zorder=2)
    soft = trav_r.copy(); soft[..., 3] = (soft[..., 3] * .55).astype(np.uint8); ax.imshow(soft, extent=ext, zorder=3, interpolation="nearest")
    for (t, rt), col in zip(rts, ["#38bdf8", "#f472b6"]):
        ax.plot(rt["lon"], rt["lat"], color=col, lw=2.4, zorder=7, path_effects=[pe.withStroke(linewidth=4.6, foreground="#05070a")])
    label_slots = [(9, 8, "left"), (9, -17, "left"), (-9, 8, "right"), (-9, -17, "right")]       # stagger labels so nearby targets stay readable
    for i, t in enumerate(tg):
        ax.scatter(t["lon"], t["lat"], s=100, c="#f59e0b", edgecolors="#05070a", linewidths=1.5, zorder=8)
        ox_, oy_, ha_ = label_slots[(i + (2 if t["lon"] > east - 0.2 * (east - west) else 0)) % 4]
        ax.annotate(f"{t['id']} {t['name']}", (t["lon"], t["lat"]), xytext=(ox_, oy_), textcoords="offset points", ha=ha_, color="white", fontsize=9, fontweight="bold", path_effects=halo, zorder=9)
    ax.scatter(blon, blat, s=140, marker="s", c="white", edgecolors="#05070a", linewidths=1.5, zorder=9)
    ax.annotate("BASE · flattest wide plain", (blon, blat), xytext=(0, -22), textcoords="offset points", ha="center", color="white", fontsize=9, fontweight="bold", path_effects=halo, zorder=9)
    ax.set_xlim(west, east); ax.set_ylim(south, north); ax.set_aspect(1 / math.cos(math.radians(spec["lat"])))
    ax.tick_params(colors="#c9d1d9", labelsize=9); [sp.set_color("#3b4650") for sp in ax.spines.values()]
    ax.set_xlabel("Longitude (°E)", color="#c9d1d9"); ax.set_ylabel("Latitude (°N)", color="#c9d1d9"); ax.grid(color="white", alpha=.12, lw=.5)
    wkm = (east - west) * M_PER_DEG * math.cos(math.radians(spec["lat"])) / 1000
    bar_km = next((v for v in [5, 10, 20, 25, 50, 100] if v <= wkm / 4), 5); bar = bar_km * 1000 / (M_PER_DEG * math.cos(math.radians(spec["lat"])))
    bx, by = west + (east - west) * .04, south + (north - south) * .04
    ax.plot([bx, bx + bar], [by, by], color="white", lw=3, zorder=10); ax.text(bx + bar / 2, by + (north - south) * .012, f"{bar_km} km", color="white", ha="center", fontsize=10, fontweight="bold", path_effects=halo, zorder=10)
    ax.annotate("N", xy=(.955, .965), xytext=(.955, .915), xycoords="axes fraction", color="white", ha="center", fontsize=13, fontweight="bold", arrowprops=dict(arrowstyle="-|>", color="white", lw=2), path_effects=halo, zorder=10)
    hnd = [Patch(color="#22c55e", alpha=.6, label="Nominal terrain"), Patch(color="#f59e0b", alpha=.7, label=f"Caution (≥{REG_SLOPE_CAUTION:.0f}° slope or rough)"), Patch(color="#dc2626", alpha=.8, label=f"No-go (≥{REG_SLOPE_BLOCK:.0f}° slope)"),
           Line2D([], [], marker="o", color="none", markerfacecolor="#f59e0b", markeredgecolor="#05070a", markersize=9, label="Relief-derived target")]
    hnd += [Line2D([], [], color=c_, lw=2.4, label=f"Route {'AB'[i]}: BASE → {rts[i][0]['id']}") for i, c_ in enumerate(["#38bdf8", "#f472b6"][:len(rts)])]
    lg = ax.legend(handles=hnd, loc="lower right", facecolor="#101318", edgecolor="#3b4650", labelcolor="white", fontsize=8.5, framealpha=.92); lg.set_zorder(11)
    fig.suptitle(f"{spec['name'].upper()} · MARSWALK PLANNING MAP", color="white", fontsize=17, fontweight="bold", x=.07, ha="left", y=.975)
    fig.text(.07, .945, f"Regional scale (463 m MOLA) · {wkm:.0f} km wide · centre {spec['lat']:.2f}°N {spec['lon']:.2f}°E", color="#929eaa", fontsize=10, ha="left")
    fig.text(.07, .125, "\n".join(f"Route {'AB'[i]}  BASE → {t['id']} {t['name']}:  {rt['km']:.1f} km  ·  about {int(rt['minutes'] // 60)} h {int(rt['minutes'] % 60):02d} min walking  ·  max slope {rt['maxSlope']:.1f}°  ·  {rt['cautionPct']:.0f}% on caution terrain" for i, (t, rt) in enumerate(rts)), color="white", fontsize=10, va="top", linespacing=1.7)
    fig.text(.07, .075, "Regional estimate from 463 m cells: local slopes are smoothed, so verify with HiRISE/CTX before use.\nWalking speed 3 km/h on easy ground; slope limits, scores and targets are concept-level assumptions.\nData: NASA MGS MOLA. No geologic map, CTX or HRSC is available for this area in the supplied files. Coordinates: Mars 2000 sphere, east-positive. Built by Marsline.", color="#929eaa", fontsize=8, va="top", linespacing=1.6)
    fig.subplots_adjust(left=.07, right=.97, top=.925, bottom=.18)
    fig.savefig(out / "mission_map.png", dpi=130, facecolor=fig.get_facecolor()); plt.close(fig)
    return meta_r, rts

built = {}
for spec in REGIONS:
    m_, rts_ = build_region(spec); built[spec["id"]] = m_
    print(f"{spec['id']:16s} grid {m_['grid']['width']}x{m_['grid']['height']} | no-go/caution/nominal "
          + " / ".join(f"{m_['traversabilityShare'][k]*100:.0f}" for k in ("nogo", "caution", "nominal")) + f" % | {len(m_['targets'])} targets | "
          + "; ".join(f"{t['id']}: {rt['km']:.1f} km {rt['minutes']/60:.1f} h" for t, rt in rts_))

nili-fossae      grid 414x384 | no-go/caution/nominal 3 / 12 / 84 % | 3 targets | T1: 13.7 km 5.0 h; T2: 18.3 km 11.8 h


syrtis-major     grid 389x385 | no-go/caution/nominal 1 / 8 / 91 % | 4 targets | T1: 11.2 km 4.4 h; T2: 12.2 km 5.0 h


isidis-planitia  grid 658x641 | no-go/caution/nominal 1 / 5 / 94 % | 3 targets | T1: 17.8 km 6.8 h; T2: 10.2 km 3.9 h


## 15 · Index of all study areas (what the app's area picker reads)

In [21]:
index = []
for d in sorted(REGIONS_DIR.iterdir()):
    mp = d / "meta.json"
    if not mp.exists(): continue
    m = json.loads(mp.read_text()); s_, w_, n_, e_ = m["gridBounds"]
    index.append(dict(id=m["id"], name=m["name"], blurb=m["blurb"], detail=m["detail"], lat=round((s_ + n_) / 2, 3), lon=round((w_ + e_) / 2, 3),
                      widthKm=round((e_ - w_) * M_PER_DEG * math.cos(math.radians((s_ + n_) / 2)) / 1000), heightKm=round((n_ - s_) * M_PER_DEG / 1000),
                      resolutionM=m["grid"]["stepM"], targets=len(m["targets"]), missionMap=m["missionMap"]))
index.sort(key=lambda r: (r["detail"] != "detailed", r["name"]))
(REGIONS_DIR / "index.json").write_text(json.dumps(index, indent=1))
pd.DataFrame(index)[["id", "name", "detail", "lat", "lon", "widthKm", "heightKm", "resolutionM", "targets"]]

,id,name,detail,lat,lon,widthKm,heightKm,resolutionM,targets
0,jezero,Jezero Crater & delta,detailed,18.200,77.299,57,59,75,7
1,isidis-planitia,Isidis Planitia,regional,12.902,87.000,297,297,463,3
2,nili-fossae,Nili Fossae,regional,21.500,74.500,178,178,463,3
3,syrtis-major,Syrtis Major · Nili Patera,regional,8.801,67.301,178,178,463,4
